# Pipeline CoT penuh — A6000 (48 GB)

Input: `data/Final/easy_clean_v3.jsonl` + `numglue_clean_v3.jsonl` (kolom `cara` sudah dibuang —
seluruh CoT datang dari teacher model).

Urutan: muat data → split holdout + dekontaminasi → **bake-off teacher** → generate penuh dengan
pemenang → rejection sampling → ChatML → statistik.

Beda dari notebook Kaggle: single GPU (`tensor_parallel_size=1`), `bfloat16` (Ampere punya bf16,
T4 tidak), tanpa workaround flashinfer, dan `aimo_hard` tidak dipakai.

**Jalankan berurutan.** Setiap tahap berat sudah resumable — kalau sesi mati, jalankan ulang selnya
dan ia melanjutkan dari checkpoint terakhir.

In [ ]:
# ── 1. Setup ───────────────────────────────────────────────────────────────
# A6000 = Ampere (sm_86): bf16 didukung, flashinfer tidak perlu di-uninstall
# (workaround itu khusus T4 compute 7.5 di notebook Kaggle).
!pip -q install "vllm>=0.6" langdetect sympy math-verify

import sys, os
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "src").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if not (REPO / "src").exists():
    !git clone -q https://github.com/henray404/FP_NLP.git
    REPO = Path("FP_NLP").resolve()
sys.path.insert(0, str(REPO))
os.chdir(REPO)

import torch
p = torch.cuda.get_device_properties(0)
print(f"REPO : {REPO}")
print(f"GPU  : {p.name}  {p.total_memory/1e9:.0f} GB  sm_{p.major}{p.minor}")
print(f"bf16 : {torch.cuda.is_bf16_supported()}")
assert torch.cuda.is_available(), "GPU tidak terdeteksi"

In [ ]:
# ── 2. Muat data v3 (kolom `cara` sudah tidak ada) ─────────────────────────
import json
from collections import Counter
from src.cot_synthesis.utils import read_jsonl

DATA = REPO / "data" / "Final"
DATASETS = ["easy_clean_v3", "numglue_clean_v3"]   # aimo_hard TIDAK dipakai

subsets = {}
for name in DATASETS:
    p = DATA / f"{name}.jsonl"
    assert p.exists(), f"tidak ketemu {p} -- jalankan judge_quality.py dulu / git pull"
    rows = read_jsonl(p)
    src = name.split("_")[0]
    for r in rows:
        r["source"] = src
    subsets[src] = rows
    has_cara = sum("cara" in r for r in rows)
    print(f"{name:20} {len(rows):5} soal   baris masih punya 'cara': {has_cara}")
    assert has_cara == 0, "kolom `cara` masih ada -- pakai file v3, bukan v2"

all_rows = [r for rows in subsets.values() for r in rows]
print(f"\nTOTAL: {len(all_rows)} soal   {Counter(r['source'] for r in all_rows)}")
print("contoh:", json.dumps(all_rows[0], ensure_ascii=False)[:160])

In [ ]:
# ── 3. Split holdout (300/subset) + train pool + dekontaminasi ─────────────
import random, re
from src.eval.make_holdout import answer_type, GRADEABLE

N_HOLDOUT, SEED = 300, 42
EVAL_DIR = REPO / "data" / "eval"; EVAL_DIR.mkdir(parents=True, exist_ok=True)

holdout, train_pool = [], []
for src, rows in subsets.items():
    for r in rows:
        r["answer_type"] = answer_type(r.get("jawaban", ""))
    gradeable = [r for r in rows if r["answer_type"] in GRADEABLE]
    random.Random(SEED).shuffle(gradeable)
    picked = gradeable[:N_HOLDOUT]
    picked_ids = {id(r) for r in picked}
    holdout += picked
    train_pool += [r for r in rows if id(r) not in picked_ids]
    print(f"{src:8} gradeable={len(gradeable):5}  holdout={len(picked)}  "
          f"pool={len(rows)-len(picked)}")

# Dekontaminasi: buang baris latih yang soalnya muncul di holdout (normalisasi spasi+kapital)
def norm(s): return re.sub(r"\s+", " ", (s or "").strip().lower())
hold_keys = {norm(r["soal"]) for r in holdout}
before = len(train_pool)
train_pool = [r for r in train_pool if norm(r["soal"]) not in hold_keys]
print(f"\ndekontaminasi: {before - len(train_pool)} baris latih dibuang (bocor ke holdout)")

def dump(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return path

dump(holdout, EVAL_DIR / "holdout.jsonl")
dump(train_pool, DATA / "train_pool.jsonl")
print(f"holdout    : {len(holdout)}  -> {EVAL_DIR/'holdout.jsonl'}")
print(f"train_pool : {len(train_pool)}  -> {DATA/'train_pool.jsonl'}")
print("komposisi holdout:", Counter(r["source"] for r in holdout))

In [ ]:
# ── 4. Kandidat teacher ────────────────────────────────────────────────────
# Semua muat bf16 di 48 GB, single GPU. Komentari yang tidak dipakai untuk hemat waktu.
#
# TIER 14B -- perbandingan terkontrol: parameter sama, 3 filosofi training berbeda.
# Ini inti eksperimennya; kalau waktu mepet, jalankan tier ini saja.
CANDIDATES = {
    "qwen3-14b":      "Qwen/Qwen3-14B",                             # ~28 GB  generalis multibahasa
    "r1-distill-14b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",   # ~28 GB  distilasi reasoning
    "openmath-14b":   "nvidia/OpenMath-Nemotron-14B",               # ~28 GB  spesialis matematika

    # TIER 7B -- kontinuitas dengan Skenario 1 paper lama
    "r1-distill-7b":  "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",    # ~15 GB  juara paper lama
    "qwen25-math-7b": "Qwen/Qwen2.5-Math-7B-Instruct",              # ~15 GB  runner-up paper lama
}

# Cadangan (butuh kuantisasi, 32B bf16 = ~64 GB > 48 GB):
#   nvidia/OpenMath-Nemotron-32B, Qwen/Qwen3-32B  -> pakai varian AWQ/GPTQ
# Cadangan model AIMO-2 juara Kaggle (subset OpenMathReasoning):
#   nvidia/OpenMath-Nemotron-14B-Kaggle
# Cadangan murah:
#   Qwen/Qwen3-8B (~16 GB), Qwen/Qwen3-4B (~8 GB), nvidia/AceMath-RL-Nemotron-7B (~15 GB)

est = {"14b": 28, "7b": 15}
for tag, m in CANDIDATES.items():
    print(f"{tag:16} {m}")
print(f"\n{len(CANDIDATES)} kandidat x {150} soal x {4} sampel = "
      f"{len(CANDIDATES)*150*4} generasi")

## Bake-off teacher

Paper lama memakai `DeepSeek-R1-Distill-Qwen-7B` (retensi 37,08%, cakupan 66,61%) karena T4 16 GB
tidak muat lebih besar. A6000 48 GB membuka pilihan, jadi teacher dipilih ulang secara empiris.

**Desain: tier 14B berisi tiga filosofi training pada jumlah parameter yang sama**, sehingga yang
diisolasi adalah keluarga model, bukan kapasitas:

| kandidat | basis | filosofi |
|---|---|---|
| `Qwen3-14B` | Qwen3 | generalis, multibahasa native (klaim 119 bahasa incl. Indonesia) |
| `DeepSeek-R1-Distill-Qwen-14B` | Qwen2.5-14B | distilasi jejak reasoning R1 |
| `OpenMath-Nemotron-14B` | Qwen2.5-14B | spesialis matematika, fine-tune OpenMathReasoning |

`OpenMath-Nemotron` penting secara metodologis: ia **berasal dari solusi juara AIMO-2**
(arXiv:2504.16891) — rujukan [5] paper ini, yaitu pendekatan yang sedang direplikasi. Memakai
teacher dari karya asalnya membuat klaim replikasi jauh lebih kuat.

### Hipotesis yang diuji (bukan sekadar cari model terbaik)

**H1 — spesialis matematika akan gagal di format/bahasa Indonesia.** Bukti dari paper ini sendiri:
Tabel XI/XII mencatat `OpenMath-Nemotron-1.5B` punya kepatuhan format hanya **0,147** (numglue) dan
**0,219** (easy) — terburuk dari semua model yang diuji. Kandidat tanpa `\boxed{}` langsung
dieliminasi `filter_solutions.py`, jadi kelemahan itu langsung memotong retensi.

**H2 — R1-Distill akan paling parah language-mixing-nya.** Literatur melaporkan model R1-Distill
justru **memperbesar** language-mixing dibanding backbone-nya ketika input bukan Inggris/Mandarin
(arXiv:2505.14815). Paper ini sudah mencatat gejalanya: ~58% langkah hasil pengisian tercampur
Inggris.

Kalau H1 dan H2 benar, pemenangnya `Qwen3-14B` — dan itu temuan yang layak dilaporkan: **untuk
distilasi CoT non-Inggris, generalis multibahasa mengalahkan spesialis matematika**, karena akurasi
matematis percuma kalau keluarannya terbuang di filter bahasa/format.

### Metrik

Selain retensi dan cakupan (seperti Skenario 1 lama), ditambah **rasio bahasa Indonesia** — metrik
yang paper belum pernah ukur. Alasannya: `to_chatml(id_only=True)` membuang CoT dominan Inggris,
jadi solusi benar tapi berbahasa Inggris **tetap terbuang**. Cakupan mentah menyesatkan.

Skor pemilihan = cakupan × rasio Indonesia.

In [ ]:
# ── 5a. Bake-off: generate kandidat, satu model per kali ───────────────────
import gc, torch
from src.cot_synthesis.generate import run_generate

COT = REPO / "data" / "cot"; COT.mkdir(parents=True, exist_ok=True)

BAKEOFF_N = 150   # soal per kandidat (cukup untuk membedakan, murah)
N_SAMPLES = 4     # kandidat solusi per soal saat bake-off (full run pakai 8)
MAX_TOKENS = 8192 # model thinking butuh ruang; R1/Qwen3 bisa panjang
BATCH = 64
DTYPE = "bfloat16"

def free_gpu():
    """Bebaskan VRAM antar model. vLLM tidak selalu melepas sendiri."""
    gc.collect()
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel
        destroy_model_parallel()
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print(f"VRAM terpakai: {torch.cuda.memory_allocated()/1e9:.1f} GB")

for tag, model in CANDIDATES.items():
    out = COT / f"bakeoff_{tag}.jsonl"
    print(f"\n{'='*60}\n{tag}  ->  {model}\n{'='*60}", flush=True)
    free_gpu()
    r = run_generate(
        DATA / "train_pool.jsonl", out,
        backend="vllm", model=model,
        n=N_SAMPLES, temperature=0.7, top_p=0.95,
        max_tokens=MAX_TOKENS, limit=BAKEOFF_N,
        tensor_parallel_size=1, batch_size=BATCH, dtype=DTYPE,
    )
    print(r)
    free_gpu()

In [ ]:
# ── 5b. Nilai kandidat bake-off: retensi, cakupan, format, RASIO INDONESIA ─
# Judge dimuat sekali untuk semua kandidat (hemat load model).
free_gpu()

from src.cot_synthesis.filter_solutions import run_filter
from src.cot_synthesis.to_chatml import is_indonesian
from src.cot_synthesis.utils import extract_boxed

bakeoff = {}
for tag, model in CANDIDATES.items():
    cand_p = COT / f"bakeoff_{tag}.jsonl"
    corr_p = COT / f"bakeoff_{tag}_correct.jsonl"
    if not cand_p.exists():
        print(f"lewati {tag}: {cand_p} tidak ada"); continue

    s = run_filter(cand_p, corr_p, judge_backend="vllm",
                   judge_model="Qwen/Qwen2.5-7B-Instruct",
                   prefilter=True, batch_size=256, resume=True)

    cands = read_jsonl(cand_p)
    n = max(len(cands), 1)
    tried = len({r["id"] for r in cands})
    bakeoff[tag] = {
        "model": model,
        "kandidat": len(cands),
        "benar": s["kept"],
        "retensi_%": round(100 * s["kept"] / n, 2),
        "cakupan_%": round(100 * s["problems_covered"] / max(tried, 1), 2),
        # H1: spesialis matematika diduga gagal format \boxed{} pada prompt Indonesia
        "format_%": round(100 * sum(extract_boxed(r["text"]) is not None for r in cands) / n, 2),
        # H2: R1-Distill diduga paling parah language-mixing
        "indonesia_%": round(100 * sum(is_indonesian(r["text"]) for r in cands) / n, 2),
    }
    print(tag, bakeoff[tag], flush=True)

# Solusi benar tapi berbahasa Inggris akan dibuang to_chatml(id_only=True),
# jadi cakupan mentah menyesatkan. Skor efektif = cakupan x rasio Indonesia.
for m in bakeoff.values():
    m["skor_efektif"] = round(m["cakupan_%"] * m["indonesia_%"] / 100, 2)

print("\n| teacher | retensi% | cakupan% | format% | Indonesia% | skor efektif |")
print("|---|---|---|---|---|---|")
for tag, m in sorted(bakeoff.items(), key=lambda kv: -kv[1]["skor_efektif"]):
    print(f"| {tag} | {m['retensi_%']} | {m['cakupan_%']} | {m['format_%']} | "
          f"{m['indonesia_%']} | {m['skor_efektif']} |")

WINNER_TAG = max(bakeoff, key=lambda t: bakeoff[t]["skor_efektif"])
WINNER = CANDIDATES[WINNER_TAG]
(COT / "bakeoff_summary.json").write_text(
    json.dumps({"hasil": bakeoff, "pemenang": WINNER_TAG}, ensure_ascii=False, indent=2),
    encoding="utf-8")
print(f"\nPEMENANG: {WINNER_TAG}  ({WINNER})")

In [ ]:
# ── 6. Generate penuh dengan teacher pemenang ──────────────────────────────
# Set manual kalau mau override hasil bake-off:
# WINNER, WINNER_TAG = "Qwen/Qwen3-14B", "qwen3-14b"
print("teacher:", WINNER)

free_gpu()
res = run_generate(
    DATA / "train_pool.jsonl",
    COT / f"candidates_{WINNER_TAG}.jsonl",
    backend="vllm", model=WINNER,
    n=8, temperature=0.7, top_p=0.95,          # resep AIMO-2, sama dengan paper
    max_tokens=MAX_TOKENS, dtype=DTYPE,
    tensor_parallel_size=1, batch_size=BATCH,
)
print(res)

In [ ]:
# ── 7. Rejection sampling penuh (judge Qwen2.5-7B-Instruct) ────────────────
# Judge dimuat SETELAH teacher dibebaskan. Resumable via .progress.
free_gpu()

from src.cot_synthesis.filter_solutions import run_filter

stats = run_filter(
    COT / f"candidates_{WINNER_TAG}.jsonl",
    COT / f"correct_{WINNER_TAG}.jsonl",
    judge_backend="vllm",
    judge_model="Qwen/Qwen2.5-7B-Instruct",
    prefilter=True,      # cek string/math_verify gratis dulu -> hemat panggilan judge
    batch_size=256,
    resume=True,
)
print(stats)
print(f"\nretensi : {100*stats['kept']/max(stats['total'],1):.2f}%")
print(f"cakupan : {100*stats['problems_covered']/len(train_pool):.2f}% "
      f"({stats['problems_covered']}/{len(train_pool)} soal)")

In [ ]:
# ── 8. Bangun ChatML (cot.jsonl + nocot.jsonl) + statistik ─────────────────
from src.cot_synthesis.to_chatml import run as build_chatml

# data/sft/train/ -- path ini yang dipakai src/training/configs/*.yaml
SFT_DIR = REPO / "data" / "sft" / "train"
cm = build_chatml(COT / f"correct_{WINNER_TAG}.jsonl", SFT_DIR,
                  best_per_problem=True,   # 1 solusi terbaik per soal
                  id_only=True)            # buang CoT dominan Inggris
print(cm)

# Statistik akhir untuk tabel paper
import statistics as st
from src.cot_synthesis.utils import read_jsonl

cot_rows = read_jsonl(SFT_DIR / "cot.jsonl")
lens = [len(r["messages"][1]["content"]) for r in cot_rows]

print(f"\nTRAIN-READY : {len(cot_rows)} pasang (cot + nocot)")
print(f"panjang CoT : median {int(st.median(lens))} char, p90 "
      f"{int(sorted(lens)[int(.9*len(lens))])}, maks {max(lens)}")
print(f"holdout     : {len(read_jsonl(EVAL_DIR / 'holdout.jsonl'))} soal")
print(f"dibuang krn bahasa Inggris : {cm.get('skipped_lang', 0)}")

In [ ]:
# ── 9. Paket hasil ──────────────────────────────────────────────────────────
import shutil

bundle = REPO / "outputs_cot_bundle"
bundle.mkdir(exist_ok=True)
for f in [SFT_DIR / "cot.jsonl", SFT_DIR / "nocot.jsonl",
          EVAL_DIR / "holdout.jsonl", DATA / "train_pool.jsonl",
          COT / f"correct_{WINNER_TAG}.jsonl", COT / "bakeoff_summary.json"]:
    if f.exists():
        shutil.copy(f, bundle / f.name)

shutil.make_archive(str(REPO / "cot_bundle"), "zip", bundle)
print("paket ->", REPO / "cot_bundle.zip")
print("isi:", [p.name for p in bundle.iterdir()])